# Moderation Policy and Decision Logic

This notebook develops the decision rules that translate machine-learning predictions into practical moderation recommendations.

The classification model produces probability scores for six harmful-content categories: toxic, severe toxic, obscene, threat, insult, and identity hate. However, model probabilities alone do not determine what action should be taken on a social-media comment.

The moderation system will therefore combine model predictions with business rules to assign one of three recommended actions:

- **ALLOW** – The comment does not contain enough evidence of harmful content to require moderation.
- **REVIEW** – The comment contains potentially harmful content and should be reviewed by a human moderator.
- **HIDE_CANDIDATE** – The model identifies strong evidence of harmful content. The comment should be prioritized for moderation and may be hidden after confirmation.

The purpose of this policy is to reduce manual moderation workload while limiting the risk of automatically suppressing legitimate comments. Human moderators will retain the ability to review and override system recommendations.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/train.csv")

validation_probabilities = np.load(
    "../data/validation_probabilities.npy"
)

print("Dataset shape:", df.shape)
print(
    "Validation probabilities shape:",
    validation_probabilities.shape
)

Dataset shape: (159571, 8)
Validation probabilities shape: (25532, 6)


In [2]:
from sklearn.model_selection import train_test_split

label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

X = df["comment_text"]
y = df[label_columns]

is_harmful = (y.sum(axis=1) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=is_harmful
)

train_is_harmful = (
    y_train.sum(axis=1) > 0
).astype(int)

X_model_train, X_validation, y_model_train, y_validation = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.20,
        random_state=42,
        stratify=train_is_harmful
    )
)

print("Validation comments:", X_validation.shape)
print("Validation labels:", y_validation.shape)
print(
    "Validation probabilities:",
    validation_probabilities.shape
)

Validation comments: (25532,)
Validation labels: (25532, 6)
Validation probabilities: (25532, 6)


In [3]:
validation_analysis = pd.DataFrame({
    "comment_text": X_validation.reset_index(drop=True)
})

for i, label in enumerate(label_columns):
    validation_analysis[f"actual_{label}"] = (
        y_validation[label].reset_index(drop=True)
    )
    validation_analysis[f"prob_{label}"] = (
        validation_probabilities[:, i]
    )

validation_analysis["actual_harmful"] = (
    validation_analysis[
        [f"actual_{label}" for label in label_columns]
    ].sum(axis=1) > 0
).astype(int)

print("Analysis table shape:", validation_analysis.shape)

validation_analysis.head()

Analysis table shape: (25532, 14)


,comment_text,actual_toxic,prob_toxic,actual_severe_toxic,prob_severe_toxic,actual_obscene,prob_obscene,actual_threat,prob_threat,actual_insult,prob_insult,actual_identity_hate,prob_identity_hate,actual_harmful
0,"""\n\n Actually a Subculture? \n\nI've seen at ...",0,0.033337,0,0.000051,0,0.009449,0,0.000118,0,0.001748,0,0.000280,0
1,I bet some weren't blind after Phillip IV gave...,0,0.005120,0,0.000016,0,0.000213,0,0.000083,0,0.000333,0,0.000088,0
2,"No, I've been responsing to is abuse of, and l...",0,0.011332,0,0.000011,0,0.000436,0,0.000045,0,0.000414,0,0.000087,0
3,central europe\n\nWho told you? I don't see a ...,0,0.000486,0,0.000006,0,0.000067,0,0.000026,0,0.000054,0,0.000030,0
4,Wtf ur a asshole no advertising pls Bot,1,0.994820,0,0.114369,1,0.981524,0,0.001919,1,0.792785,0,0.011642,1


In [4]:
final_thresholds = {
    "toxic": 0.50,
    "severe_toxic": 0.25,
    "obscene": 0.50,
    "threat": 0.20,
    "insult": 0.45,
    "identity_hate": 0.20
}

for label in label_columns:
    validation_analysis[f"predicted_{label}"] = (
        validation_analysis[f"prob_{label}"]
        >= final_thresholds[label]
    ).astype(int)

validation_analysis["predicted_harmful"] = (
    validation_analysis[
        [f"predicted_{label}" for label in label_columns]
    ].sum(axis=1) > 0
).astype(int)

In [5]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

precision = precision_score(
    validation_analysis["actual_harmful"],
    validation_analysis["predicted_harmful"]
)

recall = recall_score(
    validation_analysis["actual_harmful"],
    validation_analysis["predicted_harmful"]
)

f1 = f1_score(
    validation_analysis["actual_harmful"],
    validation_analysis["predicted_harmful"]
)

cm = confusion_matrix(
    validation_analysis["actual_harmful"],
    validation_analysis["predicted_harmful"]
)

print(f"Moderation Precision: {precision:.3f}")
print(f"Moderation Recall:    {recall:.3f}")
print(f"Moderation F1:        {f1:.3f}")
print("\nConfusion Matrix:")
print(cm)

Moderation Precision: 0.869
Moderation Recall:    0.823
Moderation F1:        0.845

Confusion Matrix:
[[22614   322]
 [  460  2136]]


In [6]:
probability_columns = [
    f"prob_{label}" for label in label_columns
]

validation_analysis["max_probability"] = (
    validation_analysis[probability_columns].max(axis=1)
)

validation_analysis[
    ["comment_text", "actual_harmful",
     "predicted_harmful", "max_probability"]
].head()

,comment_text,actual_harmful,predicted_harmful,max_probability
0,"""\n\n Actually a Subculture? \n\nI've seen at ...",0,0,0.033337
1,I bet some weren't blind after Phillip IV gave...,0,0,0.005120
2,"No, I've been responsing to is abuse of, and l...",0,0,0.011332
3,central europe\n\nWho told you? I don't see a ...,0,0,0.000486
4,Wtf ur a asshole no advertising pls Bot,1,1,0.994820


In [7]:
hide_cutoffs = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

hide_results = []

for cutoff in hide_cutoffs:
    hide_mask = (
        (validation_analysis["predicted_harmful"] == 1)
        & (validation_analysis["max_probability"] >= cutoff)
    )

    hide_count = hide_mask.sum()

    actual_harmful_count = validation_analysis.loc[
        hide_mask, "actual_harmful"
    ].sum()

    harmful_rate = (
        actual_harmful_count / hide_count
        if hide_count > 0
        else 0
    )

    hide_results.append({
        "hide_cutoff": cutoff,
        "hide_candidates": hide_count,
        "actually_harmful": actual_harmful_count,
        "false_positives": hide_count - actual_harmful_count,
        "harmful_rate": harmful_rate
    })

hide_results_df = pd.DataFrame(hide_results)

hide_results_df

,hide_cutoff,hide_candidates,actually_harmful,false_positives,harmful_rate
0,0.50,2456,2135,321,0.869300
1,0.60,2326,2070,256,0.889940
2,0.70,2148,1981,167,0.922253
3,0.80,1955,1858,97,0.950384
4,0.90,1647,1611,36,0.978142
5,0.95,1378,1365,13,0.990566


In [8]:
policy_cutoffs = [0.80, 0.90, 0.95]

policy_results = []

for cutoff in policy_cutoffs:

    action = pd.Series(
        "ALLOW",
        index=validation_analysis.index
    )

    action[
        validation_analysis["predicted_harmful"] == 1
    ] = "REVIEW"

    action[
        (validation_analysis["predicted_harmful"] == 1)
        & (validation_analysis["max_probability"] >= cutoff)
    ] = "HIDE_CANDIDATE"

    for bucket in ["ALLOW", "REVIEW", "HIDE_CANDIDATE"]:

        mask = action == bucket

        count = mask.sum()

        harmful_count = validation_analysis.loc[
            mask, "actual_harmful"
        ].sum()

        harmful_rate = (
            harmful_count / count
            if count > 0
            else 0
        )

        policy_results.append({
            "hide_cutoff": cutoff,
            "action": bucket,
            "comments": count,
            "actually_harmful": harmful_count,
            "actually_nonharmful": count - harmful_count,
            "harmful_rate": harmful_rate
        })

policy_results_df = pd.DataFrame(policy_results)

policy_results_df

,hide_cutoff,action,comments,actually_harmful,actually_nonharmful,harmful_rate
0,0.80,ALLOW,23074,460,22614,0.019936
1,0.80,REVIEW,503,278,225,0.552684
2,0.80,HIDE_CANDIDATE,1955,1858,97,0.950384
3,0.90,ALLOW,23074,460,22614,0.019936
4,0.90,REVIEW,811,525,286,0.647349
5,0.90,HIDE_CANDIDATE,1647,1611,36,0.978142
6,0.95,ALLOW,23074,460,22614,0.019936
7,0.95,REVIEW,1080,771,309,0.713889
8,0.95,HIDE_CANDIDATE,1378,1365,13,0.990566


### HIDE_CANDIDATE Threshold Selection

Several high-confidence thresholds were evaluated to determine when a harmful-content prediction should be escalated from REVIEW to HIDE_CANDIDATE.

A threshold of **0.90** was selected as the high-confidence cutoff. At this threshold:

- 23,074 comments (90.37%) were assigned to ALLOW.
- 811 comments (3.18%) were assigned to REVIEW.
- 1,647 comments (6.45%) were assigned to HIDE_CANDIDATE.
- 97.81% of HIDE_CANDIDATE comments were actually harmful.
- Only 36 of the 1,647 HIDE_CANDIDATE comments were non-harmful.

A lower cutoff of 0.80 increased the number of HIDE_CANDIDATE comments but produced 97 false positives. Raising the cutoff to 0.90 reduced this to 36 while moving uncertain cases into the human-review queue.

A cutoff of 0.95 further increased the harmful rate of HIDE_CANDIDATE comments to 99.06%, but also increased the REVIEW queue to 1,080 comments. The 0.90 cutoff therefore provides a practical balance between reducing false-positive risk and limiting manual review workload.

HIDE_CANDIDATE should be interpreted as a high-confidence moderation recommendation rather than automatic deletion. Human moderators or platform-specific rules can determine the final action.

In [9]:
hide_cutoff = 0.90

validation_analysis["moderation_action"] = "ALLOW"

validation_analysis.loc[
    validation_analysis["predicted_harmful"] == 1,
    "moderation_action"
] = "REVIEW"

validation_analysis.loc[
    (validation_analysis["predicted_harmful"] == 1)
    & (validation_analysis["max_probability"] >= hide_cutoff),
    "moderation_action"
] = "HIDE_CANDIDATE"

validation_analysis["moderation_action"].value_counts()

moderation_action
ALLOW             23074
HIDE_CANDIDATE     1647
REVIEW              811
Name: count, dtype: int64

In [10]:
total_comments = len(validation_analysis)

allow_count = (
    validation_analysis["moderation_action"] == "ALLOW"
).sum()

review_count = (
    validation_analysis["moderation_action"] == "REVIEW"
).sum()

hide_count = (
    validation_analysis["moderation_action"] == "HIDE_CANDIDATE"
).sum()

manual_attention_count = review_count + hide_count

allow_rate = allow_count / total_comments
review_rate = review_count / total_comments
hide_rate = hide_count / total_comments
manual_attention_rate = manual_attention_count / total_comments

potential_workload_reduction = allow_rate

print(f"Total comments: {total_comments:,}")
print(f"ALLOW: {allow_count:,} ({allow_rate:.2%})")
print(f"REVIEW: {review_count:,} ({review_rate:.2%})")
print(f"HIDE_CANDIDATE: {hide_count:,} ({hide_rate:.2%})")

print(
    f"\nComments requiring moderator attention: "
    f"{manual_attention_count:,} "
    f"({manual_attention_rate:.2%})"
)

print(
    f"Potential reduction in comments requiring manual inspection: "
    f"{potential_workload_reduction:.2%}"
)

Total comments: 25,532
ALLOW: 23,074 (90.37%)
REVIEW: 811 (3.18%)
HIDE_CANDIDATE: 1,647 (6.45%)

Comments requiring moderator attention: 2,458 (9.63%)
Potential reduction in comments requiring manual inspection: 90.37%


In [11]:
harmful_by_action = (
    validation_analysis[
        validation_analysis["actual_harmful"] == 1
    ]["moderation_action"]
    .value_counts()
)

total_harmful = harmful_by_action.sum()

harmful_distribution = pd.DataFrame({
    "harmful_comments": harmful_by_action,
    "percent_of_all_harmful": (
        harmful_by_action / total_harmful * 100
    )
})

harmful_distribution

,harmful_comments,percent_of_all_harmful
moderation_action,,
HIDE_CANDIDATE,1611,62.057011
REVIEW,525,20.223421
ALLOW,460,17.719569


### Moderation Policy Results

The final moderation policy combines the DistilBERT harmful-content predictions with a high-confidence cutoff of 0.90 to assign comments to ALLOW, REVIEW, or HIDE_CANDIDATE.

Across the 25,532 validation comments:

- **23,074 comments (90.37%)** were assigned to ALLOW.
- **811 comments (3.18%)** were assigned to REVIEW.
- **1,647 comments (6.45%)** were assigned to HIDE_CANDIDATE.
- REVIEW and HIDE_CANDIDATE together represent **2,458 comments (9.63%)** requiring moderator attention.

If comments assigned to ALLOW do not require routine manual inspection, the system could potentially reduce the number of comments requiring manual inspection by approximately **90.37%**.

The distribution of the 2,596 actually harmful validation comments provides an important measure of the tradeoff associated with this workload reduction:

- **1,611 harmful comments (62.06%)** were assigned to HIDE_CANDIDATE.
- **525 harmful comments (20.22%)** were assigned to REVIEW.
- **460 harmful comments (17.72%)** were incorrectly assigned to ALLOW.

Therefore, the system escalated approximately **82.28% of harmful comments** for moderator attention while allowing approximately 17.72% of harmful comments to pass without escalation.

The results demonstrate the intended human-in-the-loop design. Most comments can be separated from the moderation queue, while uncertain comments are directed to REVIEW and high-confidence harmful comments are prioritized as HIDE_CANDIDATE. However, the remaining harmful comments in the ALLOW bucket show that the system should be treated as a decision-support tool rather than a replacement for human judgment.

The estimated workload reduction represents a simulation based on the validation dataset and should not be interpreted as a guaranteed reduction in a real social-media environment. Actual performance may vary with platform, audience, comment distribution, and moderation policies.

In [12]:
test_probabilities = np.load(
    "../data/test_probabilities.npy"
)

print("Final test comments:", X_test.shape)
print("Final test labels:", y_test.shape)
print("Final test probabilities:", test_probabilities.shape)

Final test comments: (31915,)
Final test labels: (31915, 6)
Final test probabilities: (31915, 6)


In [13]:
test_analysis = pd.DataFrame({
    "comment_text": X_test.reset_index(drop=True)
})

for i, label in enumerate(label_columns):
    test_analysis[f"actual_{label}"] = (
        y_test[label].reset_index(drop=True)
    )
    test_analysis[f"prob_{label}"] = (
        test_probabilities[:, i]
    )

test_analysis["actual_harmful"] = (
    test_analysis[
        [f"actual_{label}" for label in label_columns]
    ].sum(axis=1) > 0
).astype(int)

print("Test analysis shape:", test_analysis.shape)

Test analysis shape: (31915, 14)


In [14]:
for label in label_columns:
    test_analysis[f"predicted_{label}"] = (
        test_analysis[f"prob_{label}"]
        >= final_thresholds[label]
    ).astype(int)

test_analysis["predicted_harmful"] = (
    test_analysis[
        [f"predicted_{label}" for label in label_columns]
    ].sum(axis=1) > 0
).astype(int)

test_analysis["max_probability"] = (
    test_analysis[probability_columns].max(axis=1)
)

test_analysis["moderation_action"] = "ALLOW"

test_analysis.loc[
    test_analysis["predicted_harmful"] == 1,
    "moderation_action"
] = "REVIEW"

test_analysis.loc[
    (test_analysis["predicted_harmful"] == 1)
    & (test_analysis["max_probability"] >= hide_cutoff),
    "moderation_action"
] = "HIDE_CANDIDATE"

test_analysis["moderation_action"].value_counts()

moderation_action
ALLOW             28787
HIDE_CANDIDATE     2027
REVIEW             1101
Name: count, dtype: int64

In [15]:
policy_test_summary = (
    test_analysis
    .groupby("moderation_action")["actual_harmful"]
    .agg(
        comments="count",
        actually_harmful="sum"
    )
)

policy_test_summary["actually_nonharmful"] = (
    policy_test_summary["comments"]
    - policy_test_summary["actually_harmful"]
)

policy_test_summary["harmful_rate"] = (
    policy_test_summary["actually_harmful"]
    / policy_test_summary["comments"]
)

policy_test_summary

,comments,actually_harmful,actually_nonharmful,harmful_rate
moderation_action,,,,
ALLOW,28787,549,28238,0.019071
HIDE_CANDIDATE,2027,1987,40,0.980266
REVIEW,1101,709,392,0.643960


In [16]:
test_harmful_by_action = (
    test_analysis[
        test_analysis["actual_harmful"] == 1
    ]["moderation_action"]
    .value_counts()
)

test_total_harmful = test_harmful_by_action.sum()

test_harmful_distribution = pd.DataFrame({
    "harmful_comments": test_harmful_by_action,
    "percent_of_all_harmful": (
        test_harmful_by_action
        / test_total_harmful * 100
    )
})

test_harmful_distribution

,harmful_comments,percent_of_all_harmful
moderation_action,,
HIDE_CANDIDATE,1987,61.232666
REVIEW,709,21.848998
ALLOW,549,16.918336


### Final Moderation Policy Evaluation

After the moderation policy was selected using the validation data, the frozen policy was applied to the 31,915-comment final test set without changing the category thresholds or the 0.90 HIDE_CANDIDATE cutoff.

The final test policy assigned:

- **28,787 comments (90.20%)** to ALLOW.
- **1,101 comments (3.45%)** to REVIEW.
- **2,027 comments (6.35%)** to HIDE_CANDIDATE.

Under the conservative assumption that both REVIEW and HIDE_CANDIDATE require some form of moderator attention, 3,128 comments (9.80%) would require attention. This represents a simulated **90.20% potential reduction in routine manual comment inspection** compared with reviewing every comment.

The HIDE_CANDIDATE bucket contained 2,027 comments, of which **1,987 were actually harmful and 40 were non-harmful**. Therefore, **98.03% of HIDE_CANDIDATE comments were actually harmful**.

Among the 3,245 actually harmful comments in the final test set:

- **1,987 (61.23%)** were assigned to HIDE_CANDIDATE.
- **709 (21.85%)** were assigned to REVIEW.
- **549 (16.92%)** were assigned to ALLOW.

Combined, the system escalated **83.08% of harmful comments** to either REVIEW or HIDE_CANDIDATE.

The final test results were highly consistent with the validation results. The validation HIDE_CANDIDATE harmful rate was 97.81%, compared with 98.03% on the final test set. The proportion of harmful comments that passed into ALLOW also decreased slightly from 17.72% on validation to 16.92% on the final test set.

This consistency suggests that the selected moderation policy generalizes reasonably well to unseen data. However, the 549 harmful comments assigned to ALLOW demonstrate that the system is not perfect and should remain a decision-support and prioritization tool rather than a complete replacement for human moderation.

The workload-reduction estimates are simulations based on the Jigsaw dataset and do not guarantee equivalent performance in a live social-media environment.

In [17]:
test_total_comments = len(test_analysis)

test_allow_count = (
    test_analysis["moderation_action"] == "ALLOW"
).sum()

test_attention_count = (
    test_analysis["moderation_action"] != "ALLOW"
).sum()

test_workload_reduction = (
    test_allow_count / test_total_comments
)

test_harmful_escalated = (
    test_harmful_by_action["HIDE_CANDIDATE"]
    + test_harmful_by_action["REVIEW"]
)

test_escalation_rate = (
    test_harmful_escalated / test_total_harmful
)

print(
    f"Potential routine inspection reduction: "
    f"{test_workload_reduction:.2%}"
)

print(
    f"Comments requiring moderator attention: "
    f"{test_attention_count:,}"
)

print(
    f"Harmful comments escalated: "
    f"{test_harmful_escalated:,} / "
    f"{test_total_harmful:,} "
    f"({test_escalation_rate:.2%})"
)

Potential routine inspection reduction: 90.20%
Comments requiring moderator attention: 3,128
Harmful comments escalated: 2,696 / 3,245 (83.08%)
